# Notebook 04 — Transfer Learning with EfficientNetV2

In this notebook we take a model that was pretrained on **ImageNet-1k** (1.28M images, 1000 classes) and teach it to recognise the 37 dog and cat breeds in the **Oxford-IIIT Pet** dataset. Instead of training from scratch we *reuse* the visual features the network already learned — edges, textures, shapes, parts — and only train a new tiny classifier on top. This is called **transfer learning** and it is the single most useful trick in modern computer vision.

## Learning goals
- Understand *why* transfer learning works and when to use it.
- Load EfficientNetV2-S from both **`torchvision.models`** and **`timm`** and see the trade-offs.
- Replace the pretrained 1000-class head with a 37-class head for Oxford-IIIT Pet.
- Freeze the backbone and train only the new head.
- Compare the accuracy to a scratch CNN and visualise the results.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/04_transfer_learning_efficientnetv2.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Why transfer learning?

Imagine you want to recognise 37 pet breeds and you only have ~200 training images per breed. That is *tiny* by deep-learning standards. A modern CNN has tens of millions of parameters; there simply is not enough signal in 7000 images to learn good low-level features (edges, fur texture, eye shape) **and** the high-level concept of "Abyssinian cat vs Bengal cat" from scratch.

**The key insight** — the low-level features a CNN learns from ImageNet are *extremely general*. Gabor-like edge detectors in the first layer, texture patterns in the middle, object-part detectors near the top. Cats, dogs, airplanes and bananas all share these low-level primitives. So instead of relearning them on our 7k images we **download a model that already learned them on 1.28M images** and adapt only the last layer.

| Approach | Data needed | Compute | Typical val acc on Pet |
|---|---|---|---|
| Scratch CNN (Notebook 03) | 100k+ | Days | ~25–40% |
| Transfer learning (this notebook) | 1k–10k | Minutes | ~85–93% |
| Full fine-tune (Notebook 05) | 5k–100k | Hours | ~92–96% |

Transfer learning is not magic — it just recycles the parts of the network that *any* natural-image task needs. The rest of this notebook shows how to do it.

In [ ]:
# Core imports we will use throughout the notebook
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.models as tv_models
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet

# timm is the de-facto research library for image models
import timm

print("torch       ", torch.__version__)
print("torchvision ", torchvision.__version__)
print("timm        ", timm.__version__)
print("device      ", device)

## 2. Two ecosystems for EfficientNetV2

There are two popular places to get EfficientNetV2 weights in PyTorch:

1. **`torchvision.models`** — ships with PyTorch, stable, small set of canonical weights. Great when you want zero friction.
2. **`timm`** ( *PyTorch Image Models*, by Ross Wightman) — 1000+ architectures, many variants of each, finer-grained API (`create_model`, `reset_classifier`, feature extractors). The weights here are often better (e.g. `in21k_ft_in1k` means pretrained on ImageNet-21k with 14M images, then fine-tuned on ImageNet-1k).

Both wrap **the same underlying EfficientNetV2-S architecture** — you will see the parameter count is essentially identical. The differences are in tooling and the diversity of available pretrained weights.

Rule of thumb: **use timm** unless you have a reason not to. We will do exactly that for the rest of this course.

In [ ]:
# Load EfficientNetV2-S from torchvision (1000-class ImageNet-1k weights)
tv_model = tv_models.efficientnet_v2_s(weights="DEFAULT")

# Load EfficientNetV2-S from timm, using the in21k -> in1k fine-tuned weights.
# 'in21k_ft_in1k' means: pretrained on ImageNet-21k (14M images, 21k classes),
# then fine-tuned on ImageNet-1k (1.28M images, 1000 classes).
timm_model = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k", pretrained=True)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

tv_total, tv_trainable = count_params(tv_model)
timm_total, timm_trainable = count_params(timm_model)

print(f"torchvision EfficientNetV2-S : {tv_total/1e6:6.2f} M params  (trainable {tv_trainable/1e6:.2f} M)")
print(f"timm        EfficientNetV2-S : {timm_total/1e6:6.2f} M params  (trainable {timm_trainable/1e6:.2f} M)")

In [ ]:
# Print only the top-level modules so we can see where the classifier lives.
print("=== torchvision top-level children ===")
for name, child in tv_model.named_children():
    print(f"  {name:12s} -> {type(child).__name__}")

print()
print("=== timm top-level children ===")
for name, child in timm_model.named_children():
    print(f"  {name:12s} -> {type(child).__name__}")

## 3. Replacing the classifier head

Both models currently output **1000 logits** — one per ImageNet class. We want **37 logits** — one per Pet breed. So we surgically replace the final `Linear` layer.

The "classifier head" is usually just the last `Linear(in_features, 1000)`. Its weights are ImageNet-specific and useless to us; we discard them and initialise a fresh `Linear(in_features, 37)` with random weights that we will train.

- **torchvision** — the head is `model.classifier[-1]`; we overwrite it manually.
- **timm** — has a built-in `reset_classifier(num_classes)` that does exactly this, and also knows about pooling, dropout, etc. This is one of the reasons timm is more ergonomic.

Everything *before* the head (the "backbone") keeps its pretrained weights.

In [ ]:
NUM_CLASSES = 37  # Oxford-IIIT Pet has 37 breeds

# --- torchvision way: manually replace the last Linear in model.classifier ---
in_features_tv = tv_model.classifier[-1].in_features
tv_model.classifier[-1] = nn.Linear(in_features_tv, NUM_CLASSES)
print(f"[torchvision] new head: Linear({in_features_tv}, {NUM_CLASSES})")

# --- timm way: one function call, handles pooling and dropout too ---
timm_model.reset_classifier(num_classes=NUM_CLASSES)
print(f"[timm]        new head: {timm_model.get_classifier()}")

# Quick sanity check: forward a dummy 224x224 RGB image and check the output shape
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224)
    out_tv = tv_model(dummy)
    out_timm = timm_model(dummy)
print("tv_model output  :", out_tv.shape,   "  -> (batch, NUM_CLASSES)")
print("timm_model output:", out_timm.shape, "  -> (batch, NUM_CLASSES)")

## 4. Dataset — Oxford-IIIT Pet

- 37 breeds (25 dog, 12 cat)
- ~200 images per breed, 7349 total
- Official splits: `trainval` (~3680) and `test` (~3669)

We will download the `trainval` split and carve out an 80/20 train/val split via `random_split`. The test split stays untouched so you can benchmark at the end.

### Transforms

Pretrained weights expect their input to be **normalised the same way** the network was pretrained. For ImageNet that is:

- Resize shorter side to 256, then center-crop 224×224
- Normalise with `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`

For training we add mild augmentation (`RandomResizedCrop(224)` + horizontal flip) to reduce overfitting. For validation we want a deterministic pipeline, so no augmentation.

In [ ]:
from utils.env import data_dir

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

DATA_ROOT = data_dir()
print("Downloading Oxford-IIIT Pet (first time only) to", DATA_ROOT)

full_trainval = OxfordIIITPet(root=DATA_ROOT, split="trainval",
                              target_types="category", download=True,
                              transform=train_tf)
# Same underlying files, different transform for the val subset.
val_base      = OxfordIIITPet(root=DATA_ROOT, split="trainval",
                              target_types="category", download=True,
                              transform=val_tf)
test_ds       = OxfordIIITPet(root=DATA_ROOT, split="test",
                              target_types="category", download=True,
                              transform=val_tf)

# Deterministic 80/20 split of trainval
g = torch.Generator().manual_seed(42)
n_total = len(full_trainval)
n_val = int(n_total * 0.2)
n_train = n_total - n_val
train_idx, val_idx = random_split(range(n_total), [n_train, n_val], generator=g)

class Subset(torch.utils.data.Dataset):
    """Tiny subset wrapper that keeps per-split transforms."""
    def __init__(self, ds, indices):
        self.ds = ds
        self.indices = list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        return self.ds[self.indices[i]]

train_ds = Subset(full_trainval, train_idx)
val_ds   = Subset(val_base,       val_idx)

class_names = full_trainval.classes
print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}  classes: {len(class_names)}")
print("first 5 classes:", class_names[:5])

In [ ]:
BATCH = 32

# num_workers=0 on Windows to avoid multiprocessing issues in notebooks.
# Bump it up to 2-4 on Linux / Colab for a nice speedup.
nw = 0 if os.name == "nt" else 2

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=nw, pin_memory=info.cuda_available)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=nw, pin_memory=info.cuda_available)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=nw, pin_memory=info.cuda_available)

# Peek at one batch to confirm shapes
xb, yb = next(iter(train_loader))
print("batch images :", xb.shape, xb.dtype)
print("batch labels :", yb.shape, yb.dtype, " first 5:", yb[:5].tolist())

In [ ]:
# Pull 16 samples with the VAL transform (so no random crop / flip) and show them.
from utils.plotting import show_image_grid

vis_loader = DataLoader(val_ds, batch_size=16, shuffle=True)
images, labels = next(iter(vis_loader))
titles = [class_names[y] for y in labels.tolist()]

# Un-normalise for display: x * std + mean  (reverse of T.Normalize)
mean_t = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
display_imgs = (images * std_t + mean_t).clamp(0, 1)

show_image_grid(display_imgs, titles=titles, ncols=4)

## 5. Freeze the backbone, train only the head

The classic transfer-learning recipe:

1. Freeze **every** parameter (`requires_grad = False`).
2. Unfreeze **only** the newly-created classifier head.
3. Train with a moderate LR (e.g. `1e-3`) for a few epochs.

Because only the head is trainable the optimiser has orders of magnitude fewer parameters to update, gradients only flow through the last layer, and a single epoch takes a fraction of what a full train would take. In practice this is enough to reach ~85–90% val accuracy on Pet.

We will use the **timm** model from here on — it has the cleaner API.

In [ ]:
model = timm_model.to(device)

# Step 1: freeze everything
for p in model.parameters():
    p.requires_grad = False

# Step 2: unfreeze just the classifier head
for p in model.get_classifier().parameters():
    p.requires_grad = True

# Confirm the counts
n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters     : {n_total/1e6:7.2f} M")
print(f"trainable parameters : {n_trainable/1e6:7.2f} M  ({100*n_trainable/n_total:.3f}%)")
print(f"frozen parameters    : {(n_total-n_trainable)/1e6:7.2f} M")

## 6. Train the head

We reuse `fit()` from `utils.training`, which handles:

- per-epoch train / val loops
- loss + accuracy tracking
- learning-rate scheduling hook
- early stopping
- best-model checkpointing

3 epochs is plenty for the head on Pet — the head is a single Linear layer with ~75k params, and it converges fast.

In [ ]:
from utils.training import fit
from utils.env import checkpoints_dir

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=1e-4,
)

ckpt_path = os.path.join(checkpoints_dir(), "nb04_frozen_head.pt")

history = fit(
    model, train_loader, val_loader, loss_fn, optimizer,
    epochs=3, device=device,
    scheduler=None,
    early_stopping_patience=3,
    checkpoint_path=ckpt_path,
)

print(f"best val acc = {history['best_val_acc']:.4f} at epoch {history['best_epoch']}")

In [ ]:
from utils.plotting import plot_curves
plot_curves(history)

## 7. Compare against the scratch CNN from Notebook 03

On a dataset this small, a small-from-scratch CNN typically reaches **~25–40%** val accuracy and overfits hard after ~10 epochs. We reached **~85–90%** in just **3 epochs** of head-only training.

| Model | Epochs | Val accuracy |
|---|---|---|
| Scratch CNN (Notebook 03, Pet) | 20 | ~25–40% |
| EfficientNetV2-S, frozen backbone, new head | 3 | ~85–90% |

The difference isn't the architecture — it's the 1.28M ImageNet images the pretrained backbone has already seen. Those weights contain an enormous amount of general visual knowledge and we get to borrow it for free.

## 8. Evaluation — confusion matrix

Let's look at *which* breeds the model confuses. With 37 classes the full confusion matrix is unreadable, so we'll collapse it to the 10 most-frequent breeds in the val set — enough to spot the obvious confusions (e.g. Bengal ↔ Egyptian Mau, Staffordshire Bull Terrier ↔ American Bulldog).

In [ ]:
import numpy as np
from utils.plotting import plot_confusion_matrix

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_targets.append(y)
all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

# Keep only the 10 most-frequent classes in the val set for readability
from collections import Counter
top10_classes = [cls for cls, _ in Counter(all_targets.tolist()).most_common(10)]
keep = np.isin(all_targets, top10_classes)
t_small = all_targets[keep]
p_small = all_preds[keep]

# Relabel 0..9 for the plot
remap = {c: i for i, c in enumerate(top10_classes)}
t_small = np.vectorize(remap.get)(t_small)
# predictions outside top10 map to an "other" bucket — drop them for the cm
mask = np.isin(p_small, top10_classes)
t_small = t_small[mask]
p_small = np.vectorize(remap.get)(p_small[mask])

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(t_small, p_small, labels=list(range(10)))
short_names = [class_names[c] for c in top10_classes]
plot_confusion_matrix(cm, class_names=short_names, normalize=True)

## Key Takeaways

- **Transfer learning = reuse pretrained features.** Start from ImageNet weights, replace the head, train the head.
- **On small datasets the difference vs scratch is enormous** — often 40+ percentage points of accuracy.
- **`timm` is the ergonomic choice** for EfficientNetV2. `reset_classifier()`, `get_classifier()`, `forward_features()` make everything cleaner. More weight variants too (`in21k_ft_in1k` > standard `in1k`).
- **Input pipeline must match the pretraining.** Always normalise with the same mean/std the network was trained with (ImageNet stats here).
- **Freezing the backbone = fewer trainable params, faster epochs, less overfitting** on small data — but it also caps how far you can go. Notebook 05 shows how to push further by un-freezing.

## Exercises

1. **Swap backbones.** Replace `tf_efficientnetv2_s.in21k_ft_in1k` with `tf_efficientnetv2_b0.in1k`. How many parameters now? How does val accuracy compare at 3 epochs?
2. **Different head.** Instead of a single `Linear`, make the head `Linear(D, 256) -> ReLU -> Dropout(0.2) -> Linear(256, 37)`. Does a wider head help?
3. **Smaller input size.** Re-train with 160×160 crops instead of 224×224. Measure the speed-accuracy trade-off.
4. **Test-set benchmark.** Run the final model over `test_loader` and report top-1 accuracy + a few mis-classified examples with `show_image_grid`.
5. **torchvision-only run.** Redo the training using `tv_model` instead of the timm one. Confirm the same accuracy is reachable (≈ same architecture, different wrapper).